In [1]:
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

# ================= 配置区 =================
RAW_KG_PATH = "kg.csv"
OUTPUT_CSV_PATH = "AIBL_Seed_DementiaHKG.csv"

def build_seed_and_export_csv():
    print(f"==================================================")
    print(f"🕵️ 步骤 1: 加载 PrimeKG 数据库...")
    try:
        df = pd.read_csv(RAW_KG_PATH, low_memory=False)
        all_nodes = set(df['x_name'].dropna()).union(set(df['y_name'].dropna()))
        print(f"   ✅ 成功加载 {RAW_KG_PATH}，共提取独立节点 {len(all_nodes)} 个。")
    except FileNotFoundError:
        print(f"   ❌ 找不到 {RAW_KG_PATH}，请检查路径。")
        return

    print(f"\n==================================================")
    print(f"🧬 步骤 2: 初始化医学共识核心种子 (Core Seeds)...")
    
    # 【核心升级】引入 AIBL 关注的前沿血液/脑脊液生物标志物及经典基因
    core_seeds = {
        "Alzheimer disease": "Core_Disease", 
        "Dementia": "Core_Disease",
        "Cognitive impairment": "Core_Disease",
        "APOE": "Core_Gene",
        "MAPT": "Core_Gene",     # Tau蛋白
        "APP": "Core_Gene",      # 淀粉样蛋白前体
        "PSEN1": "Core_Gene",
        "PSEN2": "Core_Gene",
        "TREM2": "Core_Gene",
        "GFAP": "Core_Protein",  # 胶质纤维酸性蛋白 (AIBL重点)
        "NEFL": "Core_Protein"   # 神经丝轻链 NfL (AIBL重点)
    }
    print(f"   定义了 {len(core_seeds)} 个核心医学种子（已注入 GFAP, NEFL, MAPT 等前沿标志物）。")

    print(f"\n==================================================")
    print(f"📊 步骤 3: 确立 AIBL 临床与表型映射字典...")
    
    # 【精准瘦身】仅保留 AIBL 数据集中存在的有效客观特征
    aibl_mapping = {
        # --- 认知与功能量表 ---
        "mmse": "Cognitive impairment",  # 简易精神状态检查
        "cdr": "Dementia",               # 临床痴呆评定
        
        # --- 深度神经心理学记忆量表 ---
        "lm_imm": "Memory impairment",   # 逻辑记忆即刻回忆
        "lm_del": "Memory impairment",   # 逻辑记忆延迟回忆
        
        # --- 基因型特征 ---
        "apoe": "APOE"                   # APOE基因型
    }
    print(f"   定义了 {len(aibl_mapping)} 个 AIBL 特征到 PrimeKG 节点的映射规则。")

    print(f"\n==================================================")
    print(f"🔍 步骤 4: 在 PrimeKG 中校验节点有效性...")
    
    valid_records = []
    
    # 校验核心种子
    print("   [校验核心种子]")
    for seed, category in core_seeds.items():
        if seed in all_nodes:
            valid_records.append({
                "AIBL_Feature": "Core_Prior_Knowledge",
                "PrimeKG_Node": seed,
                "Category": category
            })
            print(f"     ✅ 命中: {seed}")
        else:
            print(f"     ❌ 未命中: {seed} (将被丢弃)")

    # 校验 AIBL 映射
    print("   [校验 AIBL 映射特征]")
    for aibl_feat, pkg_node in aibl_mapping.items():
        if pkg_node in all_nodes:
            valid_records.append({
                "AIBL_Feature": aibl_feat,
                "PrimeKG_Node": pkg_node,
                "Category": "Clinical_Mapping"
            })
            print(f"     ✅ 命中: {aibl_feat} -> {pkg_node}")
        else:
            print(f"     ❌ 未命中: {aibl_feat} -> {pkg_node} (请检查 PrimeKG 命名)")

    print(f"\n==================================================")
    print(f"🚫 步骤 5: 附加全局黑名单规则...")
    
    # 保留防止药物网络干扰的黑名单
    valid_records.append({"AIBL_Feature": "BLACKLIST_TYPE", "PrimeKG_Node": "drug", "Category": "Rule"})
    valid_records.append({"AIBL_Feature": "BLACKLIST_TYPE", "PrimeKG_Node": "exposure", "Category": "Rule"})
    valid_records.append({"AIBL_Feature": "BLACKLIST_REL", "PrimeKG_Node": "indication", "Category": "Rule"})
    valid_records.append({"AIBL_Feature": "BLACKLIST_REL", "PrimeKG_Node": "contraindication", "Category": "Rule"})
    valid_records.append({"AIBL_Feature": "BLACKLIST_REL", "PrimeKG_Node": "off-label use", "Category": "Rule"})
    print("   已将药物节点(drug)及用药关系等 5 条黑名单规则编入导出列表。")

    print(f"\n==================================================")
    print(f"💾 步骤 6: 导出为 CSV 文件...")
    
    out_df = pd.DataFrame(valid_records)
    out_df.to_csv(OUTPUT_CSV_PATH, index=False, encoding='utf-8')
    
    print(f"   ✅ 导出完成！总计保留 {len(out_df)} 行有效配置。")
    print(f"   📄 文件已保存至: {OUTPUT_CSV_PATH}")
    print(f"==================================================")

if __name__ == "__main__":
    build_seed_and_export_csv()

🕵️ 步骤 1: 加载 PrimeKG 数据库...
   ✅ 成功加载 kg.csv，共提取独立节点 129262 个。

🧬 步骤 2: 初始化医学共识核心种子 (Core Seeds)...
   定义了 11 个核心医学种子（已注入 GFAP, NEFL, MAPT 等前沿标志物）。

📊 步骤 3: 确立 AIBL 临床与表型映射字典...
   定义了 5 个 AIBL 特征到 PrimeKG 节点的映射规则。

🔍 步骤 4: 在 PrimeKG 中校验节点有效性...
   [校验核心种子]
     ✅ 命中: Alzheimer disease
     ✅ 命中: Dementia
     ✅ 命中: Cognitive impairment
     ✅ 命中: APOE
     ✅ 命中: MAPT
     ✅ 命中: APP
     ✅ 命中: PSEN1
     ✅ 命中: PSEN2
     ✅ 命中: TREM2
     ✅ 命中: GFAP
     ✅ 命中: NEFL
   [校验 AIBL 映射特征]
     ✅ 命中: mmse -> Cognitive impairment
     ✅ 命中: cdr -> Dementia
     ✅ 命中: lm_imm -> Memory impairment
     ✅ 命中: lm_del -> Memory impairment
     ✅ 命中: apoe -> APOE

🚫 步骤 5: 附加全局黑名单规则...
   已将药物节点(drug)及用药关系等 5 条黑名单规则编入导出列表。

💾 步骤 6: 导出为 CSV 文件...
   ✅ 导出完成！总计保留 21 行有效配置。
   📄 文件已保存至: AIBL_Seed_DementiaHKG.csv
